<a href="https://colab.research.google.com/github/miffylim2308/ADALL_github/blob/main/ADALL_My_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Chapter 1. Setup and load libraries**

In [1]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries used later in Session 2
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

#**Chapter 2. Load the dataset**

In [11]:
github_raw_url = 'https://raw.githubusercontent.com/rq-goh/ADALL_github/refs/heads/main/laptop_prices_2024_sgd_TL.csv'

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

# This cell loads the laptop price dataset directly from GitHub.
# The GitHub raw URL points to the actual CSV file, not the normal GitHub preview page.
# pd.read_csv() reads the CSV file and stores it as a pandas DataFrame called df.
# df.shape shows the number of rows and columns in the dataset.
# df.head() displays the first 5 rows so we can quickly check that the data loaded correctly.

Dataset loaded successfully.
Shape: (1000, 15)


,Unnamed: 0,Brand,Model,CPU,GPU,RAM_GB,Storage_Type,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Discount_percent,Price_SGD,Brand_Discount,Member_Discount
0,0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,SSD,256,False,1.56,16.0,5.28,3207.60,80,160.38
1,1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,SSD,1024,True,1.45,14.0,6.01,2568.40,80,179.79
2,2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,SSD,2048,False,1.34,14.0,6.56,2050.80,80,143.56
3,3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,SSD,4096,True,1.18,13.3,4.62,2477.59,80,173.43
4,4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,SSD,1024,True,1.31,14.0,4.81,2626.40,80,183.85


#**Chapter 2a. Set up OpenAI**

In [3]:
# Default method: copy the prompt into the chatbot manually.
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

# Only use this section if your tutor has asked you to call the API from Colab.
# You must first save OPENAI_API_KEY in Colab Secrets.

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

# This cell prepares the notebook to use the OpenAI API, if needed.
# RUN_API_CELLS controls whether the notebook uses the API or manual chatbot mode.
# If RUN_API_CELLS is True, the notebook will try to connect to OpenAI using an API key.
# If RUN_API_CELLS is False, students can copy the prompt and paste it into ChatGPT manually.
# OPENAI_MODEL stores the model name that will be used later when sending prompts.
# client starts as None first, then becomes an OpenAI client after the API key is loaded.
# userdata.get('OPENAI_API_KEY') reads the API key saved in Colab Secrets.
# OpenAI(api_key=api_key) creates the connection object used to call the API.


OpenAI client is ready.


#**Chapter 3. LLM-assisted problem framing**

In [12]:
# PROMPT
prompt1 = f"""
You are an expert data scientist with experience in tree-based regression models.
Help me translate this business problem into a modelling objective.

Business problem: A refurbished laptop seller wants to price laptops fairly and consistently.
Dataset context: The dataset contains laptop specifications and price in SGD.

Please answer:
What should the modelling objective be?
What is the most meaningful target column?
Which metric would be easiest to explain to business users?
Who are the main stakeholders?
What are three risks or pitfalls?
"""
print('=== Prompt to send ===')
print(prompt1[:2000])

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt1
    )
    print('\n=== LLM response ===')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


=== Prompt to send ===

You are an expert data scientist with experience in tree-based regression models.
Help me translate this business problem into a modelling objective.

Business problem: A refurbished laptop seller wants to price laptops fairly and consistently.
Dataset context: The dataset contains laptop specifications and price in SGD.

Please answer:
What should the modelling objective be?
What is the most meaningful target column?
Which metric would be easiest to explain to business users?
Who are the main stakeholders?
What are three risks or pitfalls?


=== LLM response ===
### 1) What should the modelling objective be?
Build a model that **predicts the resale price (in SGD)** from laptop specifications in order to:
- **Set fair prices** for refurbished units,
- **Ensure pricing consistency** across models/brands/spec configurations,
- **Minimize prediction error** so predicted prices are close to what the market would pay for comparable specs.

In modelling terms: **super

#**Chapter 4. Quick dataset inspection before using an LLM**

In [13]:
print('Shape:', df.shape)

display(df.head())

print('\nInfo:')
display(df.info())

print('\nColumns:', df.columns.tolist())

print("\nDescribe:")
display(df.describe(include='all').transpose())

missing_values = df.isnull().sum()
print("Missing values:\n", missing_values)
duplicate_count = df.duplicated().sum()
print("\nDuplicate count:", duplicate_count)

Shape: (1000, 15)


,Unnamed: 0,Brand,Model,CPU,GPU,RAM_GB,Storage_Type,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Discount_percent,Price_SGD,Brand_Discount,Member_Discount
0,0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,SSD,256,False,1.56,16.0,5.28,3207.60,80,160.38
1,1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,SSD,1024,True,1.45,14.0,6.01,2568.40,80,179.79
2,2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,SSD,2048,False,1.34,14.0,6.56,2050.80,80,143.56
3,3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,SSD,4096,True,1.18,13.3,4.62,2477.59,80,173.43
4,4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,SSD,1024,True,1.31,14.0,4.81,2626.40,80,183.85



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1000 non-null   int64  
 1   Brand             1000 non-null   object 
 2   Model             1000 non-null   object 
 3   CPU               1000 non-null   object 
 4   GPU               1000 non-null   object 
 5   RAM_GB            1000 non-null   int64  
 6   Storage_Type      1000 non-null   object 
 7   Storage_GB        1000 non-null   int64  
 8   Touchscreen       1000 non-null   bool   
 9   Weight_kg         1000 non-null   float64
 10  Screen_Size_inch  1000 non-null   float64
 11  Discount_percent  1000 non-null   float64
 12  Price_SGD         1000 non-null   float64
 13  Brand_Discount    1000 non-null   int64  
 14  Member_Discount   1000 non-null   float64
dtypes: bool(1), float64(5), int64(4), object(5)
memory usage: 110.5+ KB


None


Columns: ['Unnamed: 0', 'Brand', 'Model', 'CPU', 'GPU', 'RAM_GB', 'Storage_Type', 'Storage_GB', 'Touchscreen', 'Weight_kg', 'Screen_Size_inch', 'Discount_percent', 'Price_SGD', 'Brand_Discount', 'Member_Discount']

Describe:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,1000.0,NaN,NaN,NaN,499.5,288.819436,0.0,249.75,499.5,749.25,999.0
Brand,1000,6,Asus,177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,1000,30,Predator Helios 300,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPU,1000,10,Intel i5-14600H,114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GPU,1000,9,NVIDIA RTX 4070,268,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RAM_GB,1000.0,NaN,NaN,NaN,53.128,44.413288,8.0,16.0,32.0,64.0,128.0
Storage_Type,1000,1,SSD,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Storage_GB,1000.0,NaN,NaN,NaN,1505.024,1380.203919,256.0,512.0,1024.0,2048.0,4096.0
Touchscreen,1000,2,False,505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Weight_kg,1000.0,NaN,NaN,NaN,2.03656,0.746477,1.0,1.34,1.97,2.68,3.5


Missing values:
 Unnamed: 0          0
Brand               0
Model               0
CPU                 0
GPU                 0
RAM_GB              0
Storage_Type        0
Storage_GB          0
Touchscreen         0
Weight_kg           0
Screen_Size_inch    0
Discount_percent    0
Price_SGD           0
Brand_Discount      0
Member_Discount     0
dtype: int64

Duplicate count: 0


#**Chapter 5. First LLM touchpoint: send only a small preview**

In [6]:
data_preview = df.head(10).to_string()
print(data_preview[:1500])

  Brand                Model                        CPU              GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
0  Acer             Aspire 5           Intel i9-14900HK  NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              3.27    2413.36               5           144.80
1  Acer              Nitro 5         AMD Ryzen 9 8900HX  AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              5.03    1773.75               5           124.16
2  Acer              Nitro 5          AMD Ryzen 5 8600H  NVIDIA RTX 4050      32          SSD        2048        False       1.34              14.0              4.41    1634.07               5            98.04
3  Acer        TravelMate P6  Intel Core Ultra 7 15500H  NVIDIA RTX 4060      16          SSD        4096         True       1.18              13.3             

In [7]:
preview_prompt = f"""
Here are the first 10 rows of a laptop pricing dataset:

{data_preview}

Questions:
1. What does each row appear to represent?
2. Which column is likely the target for a price prediction model?
3. What are 3 possible data quality checks we should perform before modelling?

Keep the answer short and practical.
"""
print('=== Prompt to send ===')
print(preview_prompt[:2000])

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=preview_prompt
    )
    print('\n=== LLM response ===')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

=== Prompt to send ===

Here are the first 10 rows of a laptop pricing dataset:

  Brand                Model                        CPU              GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
0  Acer             Aspire 5           Intel i9-14900HK  NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              3.27    2413.36               5           144.80
1  Acer              Nitro 5         AMD Ryzen 9 8900HX  AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              5.03    1773.75               5           124.16
2  Acer              Nitro 5          AMD Ryzen 5 8600H  NVIDIA RTX 4050      32          SSD        2048        False       1.34              14.0              4.41    1634.07               5            98.04
3  Acer        TravelMate P6  Intel Core Ultra 7 15500H  NVIDIA RTX 4060      1

#<font color='red'>**Chapter 7. Build the payload text**</font>

In [16]:
# Build payload text step by step.
# Payload text is a short profile of the dataset.
# It is safer and smaller than sending the full dataset to the LLM.

payload_text = ''

# 0. Business objective --- Annie added
payload_text += '=== BUSINESS OBJECTIVE ===\n'
payload_text += 'Business Goal: Predict laptop prices based on laptop specifications.\n'
payload_text += 'Questions:\n'
payload_text += '- Which features influence laptop price?\n'
payload_text += '- What data quality issues exist?\n'
payload_text += '- What preprocessing is required?\n\n'

# 1. Shape
payload_text += '=== SHAPE ===\n'
payload_text += 'Rows: ' + str(df.shape[0]) + '\n'
payload_text += 'Columns: ' + str(df.shape[1]) + '\n\n'

# 2. Sample records
payload_text += '=== SAMPLE RECORDS ===\n'
payload_text += df.head(5).to_string(index=False)
payload_text += '\n\n'

# 2. Column names and data types
payload_text += '=== COLUMNS AND DATA TYPES ===\n'
payload_text += df.dtypes.to_string()
payload_text += '\n\n'

# 3. Numeric summary
payload_text += '=== NUMERIC SUMMARY ===\n'
numeric_summary = df.describe(include='number').round(2)
payload_text += numeric_summary.to_string()
payload_text += '\n\n'

# 4. Missing values
payload_text += '=== MISSING VALUES ===\n'
missing_table = pd.DataFrame()
missing_table['missing_count'] = df.isna().sum()
missing_table['missing_pct'] = (df.isna().sum() / len(df) * 100).round(2)
payload_text += missing_table.to_string()
payload_text += '\n\n'

# 5. Unique values per column
payload_text += '=== UNIQUE VALUES PER COLUMN ===\n'
unique_table = pd.DataFrame()
unique_table['unique_count'] = df.nunique(dropna=False)
payload_text += unique_table.to_string()
payload_text += '\n\n'

# 6. Correlation between numeric columns
payload_text += '=== CORRELATION BETWEEN NUMERIC COLUMNS ===\n'
correlation_table = df.corr(numeric_only=True).round(2)
payload_text += correlation_table.to_string()
payload_text += '\n\n'

# 7. Top 10 values for categorical columns
payload_text += '=== TOP 10 VALUES FOR CATEGORICAL COLUMNS ===\n'

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_columns) == 0:
    payload_text += 'No categorical columns found.\n'
else:
    for col in categorical_columns:
        payload_text += '\nColumn: ' + col + '\n'
        payload_text += df[col].value_counts(dropna=False).head(10).to_string()
        payload_text += '\n'

payload_text += '\n'

# 8. Skewness of numeric columns
payload_text += '=== SKEWNESS OF NUMERIC COLUMNS ===\n'
skew_table = df.select_dtypes(include='number').skew().round(2)
payload_text += skew_table.to_string()
payload_text += '\n\n'

# 8. Outlier summary
payload_text += '=== OUTLIER SUMMARY (IQR METHOD) ===\n'

numeric_columns = df.select_dtypes(include='number').columns

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()
    outlier_pct = round(outlier_count / len(df) * 100, 2)

    payload_text += f'{column}: {outlier_count} ({outlier_pct}%)\n'

payload_text += '\n'

# 8. Simple warning checks
payload_text += '=== SIMPLE WARNING CHECKS ===\n'

id_like_columns = []
constant_columns = []

for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if unique_count == len(df):
        id_like_columns.append(col)

    if unique_count <= 1:
        constant_columns.append(col)
payload_text += 'Possible ID-like columns: ' + str(id_like_columns) + '\n'
payload_text += 'Constant columns: ' + str(constant_columns) + '\n'
duplicate_count = df.duplicated().sum()
duplicate_pct = round(duplicate_count / len(df) * 100, 2)
payload_text += (
    f'Duplicate rows: {duplicate_count} ({duplicate_pct}%)\n'
)

print(payload_text)

# This cell builds a short dataset profile called payload_text.
# Instead of sending the full dataset to the LLM, we send summary information only.
# The payload includes shape, column types, numeric summary, missing values, unique counts, correlations, and common category values.
# It also adds simple warning checks for possible ID-like columns, constant columns, and duplicate rows.
# This helps the LLM comment on data readiness without needing every row of the dataset.

=== BUSINESS OBJECTIVE ===
Business Goal: Predict laptop prices based on laptop specifications.
Questions:
- Which features influence laptop price?
- What data quality issues exist?
- What preprocessing is required?

=== SHAPE ===
Rows: 1000
Columns: 15

=== SAMPLE RECORDS ===
 Unnamed: 0 Brand               Model                       CPU             GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
          0  Acer            Aspire 5          Intel i9-14900HK NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              5.28    3207.60              80           160.38
          1  Acer             Nitro 5        AMD Ryzen 9 8900HX AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              6.01    2568.40              80           179.79
          2  Acer             Nitro 5         AMD Ryzen 5 8600H NVIDIA 

# <font color='red'>**Chapter 8. Ask the LLM to review data quality using payload text**


In [ ]:
quality_prompt = f"""
You are helping a data analytics student prepare a laptop pricing dataset for modelling.

Dataset profile:
{payload_text}

Task:
1. List all data quality or modelling-readiness issues.
2. Suggest action to remedy the issues.

Important:
- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=quality_prompt
    )
    print('\n=== LLM response ===')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

# This cell asks the LLM to review the dataset profile for modelling-readiness issues.
# The LLM does not receive the full dataset, only the summary stored in payload_text.
# The prompt asks for possible issues and practical remedies.
# The instructions help prevent the LLM from making unsupported assumptions or dropping columns too quickly.
# If the API client is connected, the prompt is sent automatically.
# Otherwise, students can copy the prompt and use the chatbot manually.


=== LLM response ===
Below I list the data-quality / modelling-readiness issues that are visible from the profile, followed by concrete remediation actions for each. I keep recommendations implementation-focused (what to check or do), not prescriptive about dropping columns without verification.

Issues
1. Constant feature
   - Storage_Type has only one value (SSD) → no predictive variance.
2. Possible target leakage / derived features
   - Member_Discount is very highly correlated with Price (r = 0.88).
   - Discount_percent and Brand_Discount also relate to price (Discount_percent r = -0.20).
   These may be calculated from price or otherwise contain direct information about final price.
3. Strong multicollinearity
   - Weight_kg and Screen_Size_inch are highly collinear (r = 0.92).
4. Mixed treatment of discrete numeric attributes
   - RAM_GB and Storage_GB have very few distinct values (5 each) — effectively categorical/ordinal, not continuous.
   - Screen_Size_inch only 6 levels 